# Chapter 5 - Robot Motion Models: Velocity and Odometry

Why the banana distribution?
Noise is injected into arc parameters (v, omega) and then pushed through a nonlinear
polar-to-Cartesian transform. The angular component shears the cloud.
The forward direction accumulates translational error.
The result is a banana-shaped uncertainty region.

This notebook covers:
1. Velocity model - banana distribution, per-alpha sweep
2. Odometry model - side-by-side comparison
3. Edge cases: omega near 0, v near 0, large rotation

In [ ]:
import sys
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_filters')
sys.path.insert(0, '/home/thailuu/pluto_robot/src/pluto_experiments')
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pluto_filters.motion_models.velocity_motion_model import (
    sample_motion_model_velocity, motion_model_velocity
)

def sample_velocity_model(pose, v, omega, dt, alphas, N=2000):
    a1, a2, a3, a4, a5, a6 = alphas
    xs, ys = [], []
    rng5 = np.random.default_rng(0)
    for _ in range(N):
        v_hat = v     + rng5.normal(0, np.sqrt(a1*v**2 + a2*omega**2 + 1e-12))
        w_hat = omega + rng5.normal(0, np.sqrt(a3*v**2 + a4*omega**2 + 1e-12))
        if abs(w_hat) < 1e-6:
            x2 = pose[0] + v_hat * dt * np.cos(pose[2])
            y2 = pose[1] + v_hat * dt * np.sin(pose[2])
        else:
            r  = v_hat / w_hat
            x2 = pose[0] - r*np.sin(pose[2]) + r*np.sin(pose[2] + w_hat*dt)
            y2 = pose[1] + r*np.cos(pose[2]) - r*np.cos(pose[2] + w_hat*dt)
        xs.append(x2); ys.append(y2)
    return np.array(xs), np.array(ys)

START = np.array([0.0, 0.0, 0.0])
V, OMEGA, DT5 = 0.5, 0.2, 3.0
ALPHAS_DEFAULT = (0.1, 0.01, 0.01, 0.1, 0.0, 0.0)
print("Setup complete")

In [ ]:
configs = [
    ('Low noise',     (0.01, 0.001, 0.001, 0.01, 0, 0), 'green'),
    ('Default noise', (0.10, 0.010, 0.010, 0.10, 0, 0), 'steelblue'),
    ('High noise',    (0.40, 0.050, 0.050, 0.40, 0, 0), 'crimson'),
]

# Expected final pose (noiseless)
r_arc = V / OMEGA
xf = START[0] - r_arc*np.sin(START[2]) + r_arc*np.sin(START[2]+OMEGA*DT5)
yf = START[1] + r_arc*np.cos(START[2]) - r_arc*np.cos(START[2]+OMEGA*DT5)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Banana Distribution - Velocity Motion Model', fontsize=13)

for ax, (label, alphas, color) in zip(axes, configs):
    xs, ys = sample_velocity_model(START, V, OMEGA, DT5, alphas)
    ax.scatter(xs, ys, s=3, alpha=0.4, color=color)
    ax.plot(*START[:2], 'k*', ms=12, label='Start')
    ax.plot(xf, yf, 'k+', ms=12, mew=3, label='Expected end')
    ax.set_title(label); ax.legend(fontsize=8); ax.set_aspect('equal')
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.tight_layout()
plt.savefig('ch05_banana.png', dpi=120)
plt.show()

## Per-alpha sweep: which parameter controls which aspect of the distribution?

alpha1: translational noise from translational velocity
alpha2: translational noise from rotational velocity
alpha3: rotational noise from translational velocity
alpha4: rotational noise from rotational velocity (dominant)
alpha5: final heading bias from translational velocity
alpha6: final heading bias from rotational velocity

In [ ]:
BASE   = [0.10, 0.01, 0.01, 0.10, 0.0, 0.0]
SCALE  = 8.0

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Per-alpha Sweep: What Does Each Parameter Control?', fontsize=13)
descriptions = [
    'alpha1: translational noise from v',
    'alpha2: translational noise from omega',
    'alpha3: rotational noise from v',
    'alpha4: rotational noise from omega',
    'alpha5: heading bias from v',
    'alpha6: heading bias from omega',
]

for i, (ax, desc) in enumerate(zip(axes.flat, descriptions)):
    alphas_hi = BASE.copy(); alphas_hi[i] *= SCALE
    alphas_lo = BASE.copy(); alphas_lo[i] /= SCALE
    xs_hi, ys_hi = sample_velocity_model(START, V, OMEGA, DT5, tuple(alphas_hi))
    xs_lo, ys_lo = sample_velocity_model(START, V, OMEGA, DT5, tuple(alphas_lo))
    ax.scatter(xs_lo, ys_lo, s=3, alpha=0.4, color='green',   label=f'alpha_i / {SCALE}')
    ax.scatter(xs_hi, ys_hi, s=3, alpha=0.4, color='crimson', label=f'alpha_i * {SCALE}')
    ax.plot(*START[:2], 'k*', ms=10)
    ax.set_title(desc, fontsize=9); ax.legend(fontsize=7); ax.set_aspect('equal')
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.tight_layout()
plt.savefig('ch05_alpha_sweep.png', dpi=120)
plt.show()

## Odometry Model Side-by-Side with Velocity Model

The odometry model parameterizes motion as (delta_rot1, delta_trans, delta_rot2).
Noise is injected into each odometry component independently.
This is often more accurate than the velocity model when wheel encoders are available.

In [ ]:
def sample_odometry_model(pose, odom_delta, alphas, N=2000):
    a1, a2, a3, a4 = alphas
    drot1, dtrans, drot2 = odom_delta
    xs, ys = [], []
    rng_o = np.random.default_rng(1)
    for _ in range(N):
        dr1 = drot1  - rng_o.normal(0, np.sqrt(a1*drot1**2  + a2*dtrans**2 + 1e-12))
        dt  = dtrans - rng_o.normal(0, np.sqrt(a3*dtrans**2 + a4*(drot1**2+drot2**2) + 1e-12))
        dr2 = drot2  - rng_o.normal(0, np.sqrt(a1*drot2**2  + a2*dtrans**2 + 1e-12))
        x2  = pose[0] + dt * np.cos(pose[2] + dr1)
        y2  = pose[1] + dt * np.sin(pose[2] + dr1)
        xs.append(x2); ys.append(y2)
    return np.array(xs), np.array(ys)

ODOM_DELTA  = (0.1, 1.5, 0.1)
ODOM_ALPHAS = (0.1, 0.01, 0.01, 0.1)

xs_vel,  ys_vel  = sample_velocity_model(START, 0.5, 0.05, 3.0, ALPHAS_DEFAULT)
xs_odom, ys_odom = sample_odometry_model(START, ODOM_DELTA, ODOM_ALPHAS)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Velocity Model vs Odometry Model - Noise Shape Comparison', fontsize=13)

for ax, xs, ys, title, color in [
    (axes[0], xs_vel,  ys_vel,  'Velocity model\nnoise on (v, omega)',           'steelblue'),
    (axes[1], xs_odom, ys_odom, 'Odometry model\nnoise on (rot1, trans, rot2)',  'darkorange'),
]:
    ax.scatter(xs, ys, s=4, alpha=0.4, color=color)
    ax.plot(*START[:2], 'k*', ms=12, label='Start')
    ax.set_title(title); ax.legend(fontsize=8); ax.set_aspect('equal')
    ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')
    ax.set_xlim(-1, 3); ax.set_ylim(-1.5, 1.5)

plt.tight_layout()
plt.savefig('ch05_vel_vs_odom.png', dpi=120)
plt.show()

print("Velocity model: noise on continuous (v, omega)")
print("Odometry model: noise on discrete (rot1, trans, rot2)")
print("Odometry model is typically preferred when encoders are available.")

## Edge Cases: omega near 0, v near 0, large rotation

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('Edge Cases in Velocity Motion Model', fontsize=13)

# Case 1: omega near 0 (straight line motion)
ax = axes[0]
for omega_, col, lbl in [(1e-6,'green','omega=1e-6 (straight)'),
                          (0.01,'steelblue','omega=0.01'),
                          (0.20,'crimson','omega=0.20 (arc)')]:
    xs_, ys_ = sample_velocity_model(START, 0.5, omega_, 3.0, ALPHAS_DEFAULT, N=500)
    ax.scatter(xs_, ys_, s=4, alpha=0.5, color=col, label=lbl)
ax.plot(*START[:2], 'k*', ms=12)
ax.set_title('omega near 0: straight line, no banana')
ax.legend(fontsize=8); ax.set_aspect('equal')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

# Case 2: v near 0 (rotation in place)
ax = axes[1]
for v_, col, lbl in [(0.001,'green','v=0.001 (spin in place)'),
                      (0.05,'steelblue','v=0.05'),
                      (0.30,'crimson','v=0.30')]:
    xs_, ys_ = sample_velocity_model(START, v_, 1.0, 3.0, ALPHAS_DEFAULT, N=500)
    ax.scatter(xs_, ys_, s=4, alpha=0.5, color=col, label=lbl)
ax.plot(*START[:2], 'k*', ms=12)
ax.set_title('v near 0: rotation in place')
ax.legend(fontsize=8); ax.set_aspect('equal')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

# Case 3: large rotation
ax = axes[2]
for omega_, col, lbl in [(np.pi/3,'green','omega=pi/3'),
                          (np.pi/2,'steelblue','omega=pi/2'),
                          (np.pi,  'crimson','omega=pi')]:
    xs_, ys_ = sample_velocity_model(START, 0.5, omega_, 1.0, ALPHAS_DEFAULT, N=500)
    ax.scatter(xs_, ys_, s=4, alpha=0.5, color=col, label=lbl)
ax.plot(*START[:2], 'k*', ms=12)
ax.set_title('Large rotation: arc wraps around')
ax.legend(fontsize=8); ax.set_aspect('equal')
ax.set_xlabel('x [m]'); ax.set_ylabel('y [m]')

plt.tight_layout()
plt.savefig('ch05_edge_cases.png', dpi=120)
plt.show()

## Compare, Break, Measure

In [ ]:
xs_v, ys_v   = sample_velocity_model(START, V, OMEGA, DT5, ALPHAS_DEFAULT, N=5000)
xs_o, ys_o   = sample_odometry_model(START, ODOM_DELTA, ODOM_ALPHAS, N=5000)

print("=== Compare: spread of final positions ===")
for label, xs_, ys_ in [('Velocity model', xs_v, ys_v), ('Odometry model', xs_o, ys_o)]:
    print(f"  {label}: sigma_x={xs_.std():.4f}m  sigma_y={ys_.std():.4f}m")

print("\n=== Break: negative velocity (reverse motion) ===")
xs_rev, ys_rev = sample_velocity_model(START, -0.5, 0.2, 3.0, ALPHAS_DEFAULT, N=2000)
print(f"  Forward (v=+0.5): mean_x={xs_v.mean():.3f}m")
print(f"  Reverse (v=-0.5): mean_x={xs_rev.mean():.3f}m  (should be negative)")

print("\n=== Measure: covariance determinant vs noise level ===")
print(f"{'Noise level':>15}  {'sigma_x':>10}  {'sigma_y':>10}  {'det(cov)':>12}")
print("-" * 52)
for label, alphas in [('Low',     (0.01,0.001,0.001,0.01,0,0)),
                       ('Default', ALPHAS_DEFAULT),
                       ('High',    (0.4,0.05,0.05,0.4,0,0))]:
    xs_, ys_ = sample_velocity_model(START, V, OMEGA, DT5, alphas, N=5000)
    cov = np.cov(np.vstack([xs_, ys_]))
    det = np.linalg.det(cov)
    print(f"{label:>15}  {xs_.std():>10.4f}  {ys_.std():>10.4f}  {det:>12.6f}")